手写的两个例子，FFN 和 Attention

最终会生成为 Macro Op Code

要求模拟器最后能把这里的 Code 跑起来

In [1]:
from nandmachine.config.config import NandConfig
from nandmachine.config.hbm_hbf_architecture import validate_hbm_hbf_architecture_or_raise

DEVICE_NAME = "H200_SXM"
COMPILE_MODE = "heuristic-GPU"


In [2]:
simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
    "hbf_sram_intermediate_buffer": True,
    "memory_architecture": {
        "mode": "hbm_only",
        # "mode": "csi",
        # "mode": "cli", "hbm_stacks": 2, "hbf_stacks": 3,
    },
}
validated_memory_architecture = validate_hbm_hbf_architecture_or_raise(
    simulator_config["device_name"],
    simulator_config["memory_architecture"],
)

nand_config = NandConfig(
    num_channels=8 * 4,
    num_plane=64,
    num_block=512,
    num_pages=256,
    tRead=4000,
    tWrite=40000,
    tErase=400000,
    page_size=4,
    sram_threshold=1024*20,
)

print("simulator config:", simulator_config)
print("validated memory architecture:", validated_memory_architecture)


In [3]:
from nandmachine.kernels.lieanr import LinearNandKernel


gate_marco_op_list = LinearNandKernel.lowering(32,6144,2048,16,16, nand_config)
up_marco_op_list = LinearNandKernel.lowering(32,6144,2048,16,16, nand_config)
down_macro_op_list = LinearNandKernel.lowering(32,2048,6144,16,16, nand_config)

full_list = []
full_list.extend(gate_marco_op_list)
full_list.extend(up_marco_op_list)
full_list.extend(down_macro_op_list)


In [4]:
print(full_list)

[SramPrefetch(id=1, input_ops=[], num_pages=5240832), MatMulOp(id=2, input_ops=[SramPrefetch(id=1, input_ops=[], num_pages=5240832)], dim=(32, 6144, 1706), weight_bits=16), SramPrefetchRelease(id=3, input_ops=[MatMulOp(id=2, input_ops=[SramPrefetch(id=1, input_ops=[], num_pages=5240832)], dim=(32, 6144, 1706), weight_bits=16)]), SramPrefetch(id=4, input_ops=[], num_pages=1050624), MatMulOp(id=5, input_ops=[SramPrefetch(id=4, input_ops=[], num_pages=1050624)], dim=(32, 6144, 342), weight_bits=16), SramPrefetchRelease(id=6, input_ops=[MatMulOp(id=5, input_ops=[SramPrefetch(id=4, input_ops=[], num_pages=1050624)], dim=(32, 6144, 342), weight_bits=16)]), SramPrefetch(id=7, input_ops=[], num_pages=5240832), MatMulOp(id=8, input_ops=[SramPrefetch(id=7, input_ops=[], num_pages=5240832)], dim=(32, 6144, 1706), weight_bits=16), SramPrefetchRelease(id=9, input_ops=[MatMulOp(id=8, input_ops=[SramPrefetch(id=7, input_ops=[], num_pages=5240832)], dim=(32, 6144, 1706), weight_bits=16)]), SramPrefetc

In [5]:
type(gate_marco_op_list[1])

nandmachine.commands.macro.MatMulOp